# Fund Available Summary Report — Daily Automation

**Purpose:** Build the daily `Fund Available Summary Report dd-MMM-yyyy.xlsx` from the `.eml` files that land in OneDrive.

**Folder convention (relative to this notebook):**

```
<notebook folder>/
├── Fund_Available_Summary_Report.ipynb   <- this notebook
├── Input/    <- extracted Excel attachments are placed here
└── Output/   <- final report is written here
```

**Process flow**

| # | Step | On failure |
|---|------|-----------|
| 1 | Stop if run date is Saturday / Sunday | Graceful stop (no output) |
| 2 | List `.eml` files from OneDrive | Error |
| 3 | `Funds Available as of dd MMM yyyy` -> columns A,B,C (MFUND CODE, SCHEME, GROSS) | **Error + stop** |
| 4 | MF Redemption (`FUNDING` sheet, *Total Amount* row) -> `REDEMPTION` | **Error + stop** |
| 5 | SIF Redemption (`ISIF_...`) -> `REDEMPTION` for SIF codes | Missing -> `0` shown as `-` |
| 6 | MF Additional (`Separate Statement`) -> `ADDITIONAL` | **Error + stop** |
| 7 | SIF Additional (`ISIF_...Separate Statement`) -> `ADDITIONAL` for SIF codes | Missing -> `0` shown as `-` |
| 8 | MF Daily Provision -> `SWITCH IN` / `SWITCH OUT` (Mon: Fri PM + Sun AM summed; Tue-Fri: T-1 AM) | **Error + stop** if a required file is missing |
| 9 | SIF Daily Provision (same rules) | Missing -> `0` shown as `-` (partial Monday data is used) |
| 10 | Write Excel: sort by MFUND CODE, `NET (Amount in Crs)` formula, formatting | Error |

**Global rules:** `.eml` files whose name starts with `Uploaded` are never used. Input emails must be for **T (today)**. An email may carry either a `.zip` (containing exactly one Excel file) or the Excel file directly — both are handled.

## 1. Dependencies

`xlrd` reads legacy `.xls`, `openpyxl` reads/writes `.xlsx`, `msal` + `requests` talk to OneDrive through Microsoft Graph.

In [ ]:
%pip install -q "xlrd>=2.0.1" "openpyxl>=3.1" "msal>=1.24" "requests>=2.31"

## 2. Imports

In [ ]:
from __future__ import annotations

import io
import logging
import os
import re
import sys
import zipfile
from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from decimal import ROUND_HALF_UP, Decimal, InvalidOperation
from email import policy
from email.parser import BytesParser
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple
from urllib.parse import quote
from zoneinfo import ZoneInfo

import openpyxl
import requests
import xlrd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## 3. Configuration

Everything environment-specific lives in this one cell. **Edit the `USER SETTINGS` block** (or set the matching environment variables / secrets).

* `SIF_SCHEME_CODES` — the array of SIF MFUND CODES. Any code in column A that is in this list is looked up **only** in the `ISIF_` files; every other code is looked up in the MF files.
* Secrets (`ONEDRIVE_TENANT_ID`, `ONEDRIVE_CLIENT_ID`, `ONEDRIVE_CLIENT_SECRET`) are read through `get_secret()` — never hard-code them. Hook your platform's secret store into that function.
* Set `EML_SOURCE_MODE = "local"` and `LOCAL_EML_DIR` to dry-run the notebook against a folder of `.eml` files without touching OneDrive.

In [ ]:
# ======================= USER SETTINGS =======================
SIF_SCHEME_CODES: Tuple[str, ...] = (
    "SIFEX100",        # <-- REPLACE / EXTEND with the full list of SIF MFUND CODES
)

EML_SOURCE_MODE = os.environ.get("EML_SOURCE_MODE", "onedrive")          # "onedrive" | "local"
ONEDRIVE_FOLDER_PATH = os.environ.get("ONEDRIVE_FOLDER_PATH", "FundAvailable/EML")  # folder holding the .eml files
ONEDRIVE_USER_UPN = os.environ.get("ONEDRIVE_USER_UPN")                   # e.g. "ops.mailbox@company.com"
ONEDRIVE_DRIVE_ID = os.environ.get("ONEDRIVE_DRIVE_ID")                   # alternative to the UPN (SharePoint / shared drive)
LOCAL_EML_DIR = os.environ.get("LOCAL_EML_DIR")                           # only for EML_SOURCE_MODE == "local"

TIMEZONE = "Asia/Kolkata"            # "T date" is evaluated in this timezone
RUN_DATE_OVERRIDE = None             # e.g. "2026-09-28" to re-run / test a specific day; None = today
DATED_INPUT_SUBFOLDER = True         # Input/<yyyy-mm-dd>/ per run (keeps an audit trail); False = flat Input/
# =============================================================

# ---- Business constants (change only if the source layouts change) ----
FUNDING_SHEET_NAME = "FUNDING"
HEADER_LABEL_PREFIX = "MFUND CODE"        # column-A label that marks the start of a scheme table (MFUND CODE / MFUND CODES)
TOTAL_LABEL_PREFIX = "TOTAL AMOUNT"       # column-A label of the row to read
PROVISION_CODE_COL, PROVISION_IN_COL, PROVISION_OUT_COL = 2, 5, 6   # C, F, G (0-based)
PROVISION_HEADER_CODE = "MFUND_CODE"
PROVISION_ATTACHMENT_KEY = "DAILY_PROVISION"
OUTPUT_SHEET_NAME = "Fund Available Summary Report"
OUTPUT_HEADERS = ["MFUND CODE", "SCHEME", "GROSS", "REDEMPTION", "ADDITIONAL",
                  "SWITCH IN", "SWITCH OUT", "NET (Amount in Crs)"]
CRORE_DIVISOR = Decimal(10_000_000)
EXCEL_EXTENSIONS = (".xls", ".xlsx", ".xlsm")
MAX_UNZIPPED_BYTES = 200 * 1024 * 1024    # zip-bomb guard


def get_secret(name: str, default: Optional[str] = None) -> Optional[str]:
    """Return a secret from the environment; plug your platform's secret store in here."""
    value = os.environ.get(name)
    if value:
        return value
    # Uncomment the line matching your platform:
    # from notebookutils import mssparkutils; return mssparkutils.credentials.getSecret("<key-vault-url>", name)
    # return dbutils.secrets.get(scope="<scope>", key=name)
    return default


@dataclass(frozen=True)
class Config:
    base_dir: Path
    input_dir: Path
    output_dir: Path
    source_mode: str
    onedrive_folder_path: str
    onedrive_user_upn: Optional[str]
    onedrive_drive_id: Optional[str]
    local_eml_dir: Optional[Path]
    timezone: str
    run_date_override: Optional[str]
    dated_input_subfolder: bool
    sif_scheme_codes: Tuple[str, ...]


def load_config() -> Config:
    """Resolve Input/Output next to the notebook (current working directory) and create them if missing."""
    base = Path(os.environ.get("NOTEBOOK_BASE_DIR") or Path.cwd()).resolve()
    input_dir, output_dir = base / "Input", base / "Output"
    input_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    return Config(
        base_dir=base, input_dir=input_dir, output_dir=output_dir,
        source_mode=EML_SOURCE_MODE.lower(),
        onedrive_folder_path=ONEDRIVE_FOLDER_PATH,
        onedrive_user_upn=ONEDRIVE_USER_UPN, onedrive_drive_id=ONEDRIVE_DRIVE_ID,
        local_eml_dir=Path(LOCAL_EML_DIR) if LOCAL_EML_DIR else None,
        timezone=TIMEZONE, run_date_override=RUN_DATE_OVERRIDE,
        dated_input_subfolder=DATED_INPUT_SUBFOLDER,
        sif_scheme_codes=tuple(SIF_SCHEME_CODES),
    )


CONFIG = load_config()

## 4. Logging and exception types

Distinct exception classes let the calling job tell *"business rule stop"* from *"bad data"* from *"cannot reach OneDrive"*. Any of them fails the notebook run so the scheduler can alert.

In [ ]:
def get_logger(name: str = "fund_available_report") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%Y-%m-%d %H:%M:%S"))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
        logger.propagate = False
    return logger


LOG = get_logger()


class ReportError(Exception):
    """Base class for all process errors."""


class MandatoryInputMissing(ReportError):
    """A mandatory input email / attachment is not available."""


class InputDataError(ReportError):
    """An input file exists but is malformed or ambiguous."""


class SourceAccessError(ReportError):
    """OneDrive / file-source access failed."""

## 5. Date helpers

* Month abbreviations are hard-coded, so results do **not** depend on the server locale.
* `switch_slots()` encodes the Daily Provision rule: **Monday -> Friday PM + Sunday AM**, **Tuesday-Friday -> T-1 AM**.

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def fmt_space(d: date) -> str:        # 25 Sep 2026  (Funds Available subject)
    return f"{d.day:02d} {MONTHS[d.month - 1]} {d.year}"


def fmt_dot(d: date) -> str:          # 25.09.2026   (Payment dt. / zip name)
    return f"{d.day:02d}.{d.month:02d}.{d.year}"


def fmt_dash_upper(d: date) -> str:   # 25-SEP-2026  (Daily Provision trade date)
    return f"{d.day:02d}-{MONTHS[d.month - 1].upper()}-{d.year}"


def fmt_output_date(d: date) -> str:  # 25-Sep-2026   (output file name)
    return f"{d.day:02d}-{MONTHS[d.month - 1]}-{d.year}"


def resolve_run_date(cfg: Config) -> date:
    if cfg.run_date_override:
        return date.fromisoformat(cfg.run_date_override)
    return datetime.now(ZoneInfo(cfg.timezone)).date()


def is_business_day(d: date) -> bool:
    return d.weekday() < 5            # Mon=0 ... Fri=4


def switch_slots(run_date: date) -> List[Tuple[date, str]]:
    """(date, 'AM'|'PM') Daily Provision files required for the run date."""
    if run_date.weekday() == 0:       # Monday
        return [(run_date - timedelta(days=3), "PM"), (run_date - timedelta(days=1), "AM")]
    return [(run_date - timedelta(days=1), "AM")]

## 6. Email source (OneDrive / local)

`EmlSource` is a small interface with two operations — *list* and *read bytes*. `OneDriveSource` implements it with Microsoft Graph (app-only auth, retries with back-off on 429/5xx). `LocalSource` reads a folder and is used for dry-runs and tests.

> The app registration needs the Graph **application** permission `Files.Read.All` (admin-consented). Only files directly inside `ONEDRIVE_FOLDER_PATH` are listed (no sub-folders).

In [ ]:
@dataclass(frozen=True)
class SourceFile:
    name: str
    ref: str               # OneDrive item id, or local path
    modified: datetime     # timezone-aware


class EmlSource(ABC):
    @abstractmethod
    def list_files(self) -> List[SourceFile]: ...

    @abstractmethod
    def read_bytes(self, file: SourceFile) -> bytes: ...


class LocalSource(EmlSource):
    def __init__(self, folder: Path):
        if not folder or not Path(folder).is_dir():
            raise SourceAccessError(f"Local eml folder not found: {folder}")
        self._folder = Path(folder)

    def list_files(self) -> List[SourceFile]:
        return [SourceFile(p.name, str(p), datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc))
                for p in self._folder.iterdir() if p.is_file() and p.suffix.lower() == ".eml"]

    def read_bytes(self, file: SourceFile) -> bytes:
        return Path(file.ref).read_bytes()


class OneDriveSource(EmlSource):
    GRAPH = "https://graph.microsoft.com/v1.0"
    SCOPE = ["https://graph.microsoft.com/.default"]

    def __init__(self, tenant_id: str, client_id: str, client_secret: str, folder_path: str,
                 user_upn: Optional[str] = None, drive_id: Optional[str] = None, timeout: int = 60):
        if not (user_upn or drive_id):
            raise SourceAccessError("Set ONEDRIVE_USER_UPN or ONEDRIVE_DRIVE_ID.")
        if not (tenant_id and client_id and client_secret):
            raise SourceAccessError("Missing ONEDRIVE_TENANT_ID / ONEDRIVE_CLIENT_ID / ONEDRIVE_CLIENT_SECRET.")
        import msal  # imported lazily so local mode does not need it
        self._app = msal.ConfidentialClientApplication(
            client_id, authority=f"https://login.microsoftonline.com/{tenant_id}", client_credential=client_secret)
        self._drive_root = (f"{self.GRAPH}/drives/{drive_id}" if drive_id
                            else f"{self.GRAPH}/users/{quote(user_upn)}/drive")
        self._folder = folder_path.strip("/")
        self._timeout = timeout
        self._session = self._build_session()

    @staticmethod
    def _build_session() -> requests.Session:
        retry = Retry(total=5, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504),
                      allowed_methods=("GET",), respect_retry_after_header=True)
        session = requests.Session()
        session.mount("https://", HTTPAdapter(max_retries=retry))
        return session

    def _auth_headers(self) -> Dict[str, str]:
        result = self._app.acquire_token_for_client(scopes=self.SCOPE)   # msal caches / refreshes tokens
        token = result.get("access_token")
        if not token:
            raise SourceAccessError(f"Graph authentication failed: {result.get('error')} - {result.get('error_description')}")
        return {"Authorization": f"Bearer {token}"}

    def _get(self, url: str, params: Optional[dict] = None) -> requests.Response:
        try:
            response = self._session.get(url, headers=self._auth_headers(), params=params, timeout=self._timeout)
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            raise SourceAccessError(f"OneDrive request failed: {url} ({exc})") from exc

    def list_files(self) -> List[SourceFile]:
        url = (f"{self._drive_root}/root:/{quote(self._folder)}:/children" if self._folder
               else f"{self._drive_root}/root/children")
        params: Optional[dict] = {"$top": "200", "$select": "id,name,file,lastModifiedDateTime"}
        files: List[SourceFile] = []
        while url:
            payload = self._get(url, params).json()
            params = None                                   # nextLink already carries the query
            for item in payload.get("value", []):
                if "file" in item and item["name"].lower().endswith(".eml"):
                    modified = datetime.fromisoformat(item["lastModifiedDateTime"].replace("Z", "+00:00"))
                    files.append(SourceFile(item["name"], item["id"], modified))
            url = payload.get("@odata.nextLink")
        return files

    def read_bytes(self, file: SourceFile) -> bytes:
        return self._get(f"{self._drive_root}/items/{file.ref}/content").content


def build_source(cfg: Config) -> EmlSource:
    if cfg.source_mode == "local":
        return LocalSource(cfg.local_eml_dir)
    if cfg.source_mode == "onedrive":
        return OneDriveSource(
            tenant_id=get_secret("ONEDRIVE_TENANT_ID"), client_id=get_secret("ONEDRIVE_CLIENT_ID"),
            client_secret=get_secret("ONEDRIVE_CLIENT_SECRET"), folder_path=cfg.onedrive_folder_path,
            user_upn=cfg.onedrive_user_upn, drive_id=cfg.onedrive_drive_id)
    raise SourceAccessError(f"Unknown EML_SOURCE_MODE: {cfg.source_mode!r}")

## 7. Locating the right `.eml` file

Matching is on the **file name**, anchored at the start (optionally after `RE:` / `FW:`). Anchoring is what keeps the families apart:

* `Uploaded Redemption Payout - ...` never matches (also blocked explicitly).
* `Redemption Payout - Funding Statement ...` does **not** match `ISIF_Redemption Payout - Funding Statement ...` (and vice versa), because the name must *begin* with the literal.
* A trailing digit guard stops `25.09.2026` matching `25.09.20261`.

If several files match (e.g. an email was re-sent) the most recently modified one is used and a warning is logged.

In [ ]:
_PREFIX_RE = r"^\s*(?:(?:re|fw|fwd)\s*:\s*)*"


def build_name_pattern(literal: str, digit_guard: bool = True) -> re.Pattern:
    return re.compile(_PREFIX_RE + re.escape(literal) + (r"(?!\d)" if digit_guard else ""), re.IGNORECASE)


def pat_funds_available(d: date) -> re.Pattern:
    return build_name_pattern(f"Funds Available as of {fmt_space(d)}")


def pat_redemption(d: date, sif: bool) -> re.Pattern:
    return build_name_pattern(f"{'ISIF_' if sif else ''}Redemption Payout - Funding Statement for the Payment dt. {fmt_dot(d)}")


def pat_additional(d: date, sif: bool) -> re.Pattern:
    return build_name_pattern(f"{'ISIF_' if sif else ''}Redemption Payout - Separate Statement for the Payment dt. {fmt_dot(d)}")


def pat_provision(d: date, am_pm: str, sif: bool) -> re.Pattern:
    literal = (f"{'ISIF ' if sif else ''}Daily Provision and Net Off Intra Switches file "
               f"for the trade dated {fmt_dash_upper(d)}.{am_pm}")
    return build_name_pattern(literal, digit_guard=False)   # real names carry a timestamp suffix after AM/PM


def _is_uploaded(name: str) -> bool:
    stripped = re.sub(_PREFIX_RE, "", name, flags=re.IGNORECASE)
    return stripped.lower().startswith("uploaded")


def find_eml(files: Sequence[SourceFile], pattern: re.Pattern) -> Optional[SourceFile]:
    matches = [f for f in files if not _is_uploaded(f.name) and pattern.search(f.name)]
    if not matches:
        return None
    if len(matches) > 1:
        LOG.warning("%d emails match %s - using the latest: %s", len(matches), pattern.pattern[-60:],
                    [m.name for m in matches])
    return max(matches, key=lambda f: f.modified)

## 8. Attachment extraction (zip or direct Excel)

Rules implemented in `extract_single_excel()`:

1. Collect every attachment (including nested `message/rfc822`).
2. If the email has a **zip**, unzip it in memory (path-traversal safe — only the base name is used) and take the Excel file(s) inside. Otherwise take the Excel attachment(s) directly.
3. An optional `name_filter` narrows the candidates (used for `DAILY_PROVISION` — those emails carry several attachments).
4. Exactly **one** Excel file must remain, otherwise `InputDataError`. The file is saved into the `Input` folder.

In [ ]:
def _safe_basename(name: str) -> str:
    return re.split(r"[\\/]", name)[-1].strip()


def iter_attachments(eml_bytes: bytes) -> List[Tuple[str, bytes]]:
    message = BytesParser(policy=policy.default).parsebytes(eml_bytes)
    attachments: List[Tuple[str, bytes]] = []
    for part in message.walk():
        if part.is_multipart():
            continue
        filename = part.get_filename()
        if not filename:
            continue
        payload = part.get_payload(decode=True)
        if payload:
            attachments.append((_safe_basename(filename), payload))
    return attachments


def _excel_members_of_zip(zip_bytes: bytes) -> List[Tuple[str, bytes]]:
    found: List[Tuple[str, bytes]] = []
    try:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
            if sum(i.file_size for i in archive.infolist()) > MAX_UNZIPPED_BYTES:
                raise InputDataError("Zip attachment expands beyond the allowed size.")
            for info in archive.infolist():
                base = _safe_basename(info.filename)
                if info.is_dir() or "__MACOSX" in info.filename or base.startswith(("~$", ".")):
                    continue
                if base.lower().endswith(EXCEL_EXTENSIONS):
                    found.append((base, archive.read(info)))
    except zipfile.BadZipFile as exc:
        raise InputDataError(f"Corrupt zip attachment: {exc}") from exc
    return found


def extract_single_excel(eml_bytes: bytes, dest_dir: Path, label: str,
                         name_filter: Optional[Callable[[str], bool]] = None) -> Path:
    attachments = iter_attachments(eml_bytes)
    candidates: List[Tuple[str, bytes]] = []
    for name, data in attachments:
        if name.lower().endswith(".zip"):
            candidates.extend(_excel_members_of_zip(data))
    origin = "zip"
    if not candidates:
        origin = "direct attachment"
        candidates = [(n, d) for n, d in attachments if n.lower().endswith(EXCEL_EXTENSIONS)]
    if name_filter:
        candidates = [(n, d) for n, d in candidates if name_filter(n)]
    if not candidates:
        raise InputDataError(f"{label}: no Excel file found in the email attachments "
                             f"(attachments seen: {[n for n, _ in attachments]}).")
    if len(candidates) > 1:
        raise InputDataError(f"{label}: expected exactly one Excel file but found {[n for n, _ in candidates]}.")
    name, data = candidates[0]
    dest_dir.mkdir(parents=True, exist_ok=True)
    target = dest_dir / name
    target.write_bytes(data)
    LOG.info("%s: saved %s (%s, %s bytes)", label, target.name, origin, f"{len(data):,}")
    return target


def acquire_excel(source: EmlSource, index: Sequence[SourceFile], pattern: re.Pattern, dest_dir: Path,
                  label: str, mandatory: bool,
                  name_filter: Optional[Callable[[str], bool]] = None) -> Optional[Path]:
    """Find the email, download it, extract its Excel file. Missing + mandatory -> MandatoryInputMissing."""
    match = find_eml(index, pattern)
    if match is None:
        if mandatory:
            raise MandatoryInputMissing(f"{label}: no email found matching /{pattern.pattern}/")
        LOG.warning("%s: email not found - values will default to 0 (shown as '-').", label)
        return None
    LOG.info("%s: using email '%s'", label, match.name)
    return extract_single_excel(source.read_bytes(match), dest_dir, label, name_filter)

## 9. Excel reading and amount handling

* `read_workbook()` sniffs the real file type from the first bytes (a file named `.XLS` may actually be `.xlsx`) and returns every sheet as a grid of `CellValue(value, fmt)`.
* Amounts are handled with `Decimal` (built from the shortest round-trip text of the stored number), so **no rounding or precision change** is introduced.
* `to_decimal()` converts `(8134535.56)` -> `-8134535.56`; plain numbers stay untouched.
* `decimals_of()` keeps the source's decimal places (from the source number format when available, else from the value; minimum 2).
* Zero results are written with a number format that **displays `-`**.

In [ ]:
@dataclass(frozen=True)
class CellValue:
    value: object = None
    fmt: Optional[str] = None


Grid = List[List[CellValue]]
EMPTY = CellValue()


@dataclass(frozen=True)
class Amount:
    value: Decimal
    decimals: int = 2

    def __add__(self, other: "Amount") -> "Amount":
        return Amount(self.value + other.value, max(self.decimals, other.decimals))


ZERO = Amount(Decimal(0), 2)


def _sniff_excel_type(path: Path) -> str:
    with open(path, "rb") as fh:
        head = fh.read(8)
    if head.startswith(b"PK\x03\x04"):
        return "xlsx"
    if head.startswith(b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"):
        return "xls"
    raise InputDataError(f"{path.name}: not a recognised Excel file (header bytes {head!r}).")


def _read_xls(path: Path) -> Dict[str, Grid]:
    try:
        book = xlrd.open_workbook(str(path), formatting_info=True)
    except Exception:                                       # formatting info unavailable -> values only
        book = xlrd.open_workbook(str(path), formatting_info=False)
    has_fmt = bool(getattr(book, "xf_list", None))
    sheets: Dict[str, Grid] = {}
    for sheet in book.sheets():
        grid: Grid = []
        for r in range(sheet.nrows):
            row: List[CellValue] = []
            for c in range(sheet.ncols):
                cell = sheet.cell(r, c)
                if cell.ctype in (xlrd.XL_CELL_EMPTY, xlrd.XL_CELL_BLANK, xlrd.XL_CELL_ERROR):
                    row.append(EMPTY)
                    continue
                fmt = None
                if has_fmt and cell.xf_index is not None:
                    fmt = book.format_map[book.xf_list[cell.xf_index].format_key].format_str
                row.append(CellValue(cell.value, fmt))
            grid.append(row)
        sheets[sheet.name] = grid
    return sheets


def _read_xlsx(path: Path) -> Dict[str, Grid]:
    book = openpyxl.load_workbook(path, data_only=True)      # cached values (formulas not evaluated)
    sheets: Dict[str, Grid] = {}
    for ws in book.worksheets:
        sheets[ws.title] = [[CellValue(c.value, getattr(c, "number_format", None)) for c in row]
                            for row in ws.iter_rows()]
    book.close()
    return sheets


def read_workbook(path: Path) -> Dict[str, Grid]:
    kind = _sniff_excel_type(path)
    sheets = _read_xls(path) if kind == "xls" else _read_xlsx(path)
    if not sheets:
        raise InputDataError(f"{path.name}: workbook has no sheets.")
    return sheets


def cell_at(grid: Grid, row: int, col: int) -> CellValue:
    if 0 <= row < len(grid) and 0 <= col < len(grid[row]):
        return grid[row][col]
    return EMPTY


def norm_text(value: object) -> str:
    """Upper-case, whitespace-collapsed text for label / code comparison."""
    if value is None:
        return ""
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    return re.sub(r"\s+", " ", str(value)).strip().upper()


def to_decimal(value: object, where: str = "") -> Decimal:
    """Exact Decimal from a number or text. '(1,234.50)' -> -1234.50 ; blank / '-' -> 0."""
    if value is None:
        return Decimal(0)
    if isinstance(value, bool):
        raise InputDataError(f"Boolean found where an amount was expected {where}")
    if isinstance(value, int):
        return Decimal(value)
    if isinstance(value, float):
        return Decimal(repr(value))
    text = str(value).strip().replace(",", "")
    if text in ("", "-", "--"):
        return Decimal(0)
    negative = text.startswith("(") and text.endswith(")")
    if negative:
        text = text[1:-1].strip()
    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise InputDataError(f"Cannot read '{value}' as an amount {where}") from exc
    return -number if negative else number


_FMT_DECIMALS_RE = re.compile(r"0\.(0+)")


def decimals_of(cell: CellValue, number: Decimal) -> int:
    if cell.fmt:
        match = _FMT_DECIMALS_RE.search(cell.fmt)
        if match:
            return len(match.group(1))
    exponent = number.as_tuple().exponent
    return max(2, -exponent) if isinstance(exponent, int) else 2


def cell_to_amount(cell: CellValue, where: str = "") -> Amount:
    number = to_decimal(cell.value, where)
    return Amount(number, decimals_of(cell, number))

## 10. Parsers for each input file

**Funds Available** — single sheet, columns A/B/C = `MFUND CODE`, `SCHEME`, `GROSS`. Bracketed GROSS values become negative.

**Funding statements** (`FUNDING` sheet, MF/SIF Redemption and Additional) — the sheet contains a table whose column-A label is `MFUND CODE(S)`; on that row, columns B onward hold the scheme codes. The `Total Amount` row below it holds the values. The parser builds a `scheme code -> Amount` dictionary once per file (so lookups are O(1)). A code that is not in the dictionary later becomes `0` (displayed `-`).

**Daily Provision** — column C = `MFUND_CODE`, F = `SWITCH_IN`, G = `SWITCH_OUT`. If a code appears on several rows the values are summed. On Mondays the two files (Friday PM + Sunday AM) are summed per code.

In [ ]:
@dataclass(frozen=True)
class FundRow:
    code: str
    scheme: str
    gross: Amount


def parse_funds_available(path: Path) -> List[FundRow]:
    sheets = read_workbook(path)
    if len(sheets) != 1:
        LOG.warning("%s: expected 1 sheet, found %d - reading the first.", path.name, len(sheets))
    grid = next(iter(sheets.values()))
    header_row = next((r for r in range(min(len(grid), 20))
                       if norm_text(cell_at(grid, r, 0).value).startswith(HEADER_LABEL_PREFIX)), None)
    if header_row is None:
        raise InputDataError(f"{path.name}: header row 'MFUND CODE / SCHEME / GROSS' not found in the first 20 rows.")
    for col, expected in ((1, "SCHEME"), (2, "GROSS")):
        if norm_text(cell_at(grid, header_row, col).value) != expected:
            LOG.warning("%s: column %s header is '%s', expected '%s'.", path.name, get_column_letter(col + 1),
                        cell_at(grid, header_row, col).value, expected)
    rows: List[FundRow] = []
    for r in range(header_row + 1, len(grid)):
        code_cell = cell_at(grid, r, 0)
        code = "" if code_cell.value is None else str(code_cell.value).strip()
        if not code:
            continue
        scheme = cell_at(grid, r, 1).value
        rows.append(FundRow(code, "" if scheme is None else str(scheme).strip(),
                            cell_to_amount(cell_at(grid, r, 2), f"({path.name} row {r + 1}, GROSS)")))
    if not rows:
        raise InputDataError(f"{path.name}: no data rows found.")
    LOG.info("Funds Available: %d scheme rows read.", len(rows))
    return rows


def parse_funding_statement(path: Path, label: str) -> Dict[str, Amount]:
    """Return {normalised scheme code: Total Amount} from the FUNDING sheet."""
    sheets = read_workbook(path)
    sheet_name = next((n for n in sheets if norm_text(n) == FUNDING_SHEET_NAME), None)
    if sheet_name is None:
        raise InputDataError(f"{label}: sheet '{FUNDING_SHEET_NAME}' not found in {path.name} (sheets: {list(sheets)}).")
    grid = sheets[sheet_name]
    header_rows = [r for r in range(len(grid)) if norm_text(cell_at(grid, r, 0).value).startswith(HEADER_LABEL_PREFIX)]
    if not header_rows:
        raise InputDataError(f"{label}: no '{HEADER_LABEL_PREFIX}' label found in column A of {path.name}.")
    total_rows = [r for r in range(len(grid)) if norm_text(cell_at(grid, r, 0).value).startswith(TOTAL_LABEL_PREFIX)]
    width = max((len(row) for row in grid), default=0)
    totals: Dict[str, Amount] = {}
    for i, header_row in enumerate(header_rows):
        next_header = header_rows[i + 1] if i + 1 < len(header_rows) else len(grid)
        total_row = next((r for r in total_rows if header_row < r < next_header), None)
        if total_row is None:
            raise InputDataError(f"{label}: no '{TOTAL_LABEL_PREFIX}' row below the table starting at row {header_row + 1}.")
        for col in range(1, width):
            code = norm_text(cell_at(grid, header_row, col).value)
            if not code:
                continue
            if code in totals:
                LOG.warning("%s: scheme code %s appears more than once - keeping the first.", label, code)
                continue
            totals[code] = cell_to_amount(cell_at(grid, total_row, col),
                                          f"({path.name}, {code}, row {total_row + 1})")
    LOG.info("%s: %d scheme totals read from %s.", label, len(totals), path.name)
    return totals


def parse_daily_provision(path: Path, label: str) -> Dict[str, Tuple[Amount, Amount]]:
    """Return {normalised MFUND_CODE: (SWITCH_IN, SWITCH_OUT)}; duplicate codes are summed."""
    sheets = read_workbook(path)
    grid, header_row = None, None
    for candidate in sheets.values():
        header_row = next((r for r in range(min(len(candidate), 20))
                           if norm_text(cell_at(candidate, r, PROVISION_CODE_COL).value) == PROVISION_HEADER_CODE), None)
        if header_row is not None:
            grid = candidate
            break
    if grid is None:
        raise InputDataError(f"{label}: header '{PROVISION_HEADER_CODE}' not found in column C of {path.name}.")
    for col, expected in ((PROVISION_IN_COL, "SWITCH_IN"), (PROVISION_OUT_COL, "SWITCH_OUT")):
        if norm_text(cell_at(grid, header_row, col).value) != expected:
            LOG.warning("%s: column %s header is '%s', expected '%s'.", label, get_column_letter(col + 1),
                        cell_at(grid, header_row, col).value, expected)
    result: Dict[str, Tuple[Amount, Amount]] = {}
    for r in range(header_row + 1, len(grid)):
        code = norm_text(cell_at(grid, r, PROVISION_CODE_COL).value)
        if not code:
            continue
        where = f"({path.name} row {r + 1})"
        switch_in = cell_to_amount(cell_at(grid, r, PROVISION_IN_COL), where)
        switch_out = cell_to_amount(cell_at(grid, r, PROVISION_OUT_COL), where)
        previous = result.get(code)
        result[code] = (switch_in, switch_out) if previous is None else (previous[0] + switch_in, previous[1] + switch_out)
    LOG.info("%s: %d scheme rows read from %s.", label, len(result), path.name)
    return result


def merge_switches(parts: Sequence[Dict[str, Tuple[Amount, Amount]]]) -> Dict[str, Tuple[Amount, Amount]]:
    """Sum SWITCH_IN / SWITCH_OUT per code across files (Monday: Friday PM + Sunday AM)."""
    merged: Dict[str, Tuple[Amount, Amount]] = {}
    for part in parts:
        for code, (s_in, s_out) in part.items():
            prev = merged.get(code)
            merged[code] = (s_in, s_out) if prev is None else (prev[0] + s_in, prev[1] + s_out)
    return merged

## 11. Business logic — assemble report rows

* Rows are sorted **ascending by MFUND CODE** (case-insensitive, stable).
* A code in `SIF_SCHEME_CODES` uses the SIF dictionaries; every other code uses the MF dictionaries.
* Not found -> `ZERO` (written as `0`, displayed `-`).
* `compute_net_crores()` mirrors the Excel formula and is used for logging / verification:
  `NET = ROUND((GROSS - REDEMPTION - ADDITIONAL + SWITCH IN - SWITCH OUT) / 1,00,00,000, 2)` — e.g. `4,65,00,546.43` -> `4.65`.

In [ ]:
@dataclass(frozen=True)
class ReportRow:
    code: str
    scheme: str
    gross: Amount
    redemption: Amount
    additional: Amount
    switch_in: Amount
    switch_out: Amount
    is_sif: bool


def build_report_rows(fund_rows: Sequence[FundRow], sif_codes: Sequence[str],
                      redemption_mf: Dict[str, Amount], redemption_sif: Dict[str, Amount],
                      additional_mf: Dict[str, Amount], additional_sif: Dict[str, Amount],
                      switches_mf: Dict[str, Tuple[Amount, Amount]],
                      switches_sif: Dict[str, Tuple[Amount, Amount]]) -> List[ReportRow]:
    sif_set = {norm_text(c) for c in sif_codes}
    if not sif_set:
        LOG.warning("SIF_SCHEME_CODES is empty - every scheme will be treated as an MF scheme.")
    missing: Dict[str, List[str]] = {"REDEMPTION": [], "ADDITIONAL": [], "SWITCH": []}
    rows: List[ReportRow] = []
    for fund in sorted(fund_rows, key=lambda f: (f.code.upper(), f.code)):
        key = norm_text(fund.code)
        is_sif = key in sif_set
        red_map, add_map, sw_map = ((redemption_sif, additional_sif, switches_sif) if is_sif
                                    else (redemption_mf, additional_mf, switches_mf))
        redemption, additional, switch = red_map.get(key), add_map.get(key), sw_map.get(key)
        if redemption is None: missing["REDEMPTION"].append(fund.code)
        if additional is None: missing["ADDITIONAL"].append(fund.code)
        if switch is None: missing["SWITCH"].append(fund.code)
        rows.append(ReportRow(fund.code, fund.scheme, fund.gross, redemption or ZERO, additional or ZERO,
                              switch[0] if switch else ZERO, switch[1] if switch else ZERO, is_sif))
    for column, codes in missing.items():
        if codes:
            LOG.warning("%s: %d scheme(s) not found in the input file(s) -> 0: %s", column, len(codes), codes)
    return rows


def compute_net_crores(row: ReportRow, divisor: Decimal = CRORE_DIVISOR) -> Decimal:
    raw = (row.gross.value - row.redemption.value - row.additional.value
           + row.switch_in.value - row.switch_out.value)
    return (raw / divisor).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

## 12. Excel writer

* Columns A-G hold the values; column **H is a live Excel formula**: `=ROUND((C-D-E+F-G)/10000000,2)`.
* Amount columns keep the source decimals; REDEMPTION / ADDITIONAL / SWITCH IN / SWITCH OUT use a number format whose zero section shows `-`.
* The file is written to a temporary name and then atomically renamed, so a failed run never leaves a half-written report.
* `verify_output()` re-opens the file and checks sheet name, headers, row count and formulas.

> `openpyxl` cannot calculate formulas, so column H has no cached value until the file is opened in Excel (which calculates on load). If a downstream program must read column H without opening Excel, recalculate the file with LibreOffice headless after writing.

In [ ]:
FONT = Font(name="Arial", size=10)
HEADER_FONT = Font(name="Arial", size=10, bold=True)
HEADER_FILL = PatternFill("solid", start_color="D9E1F2")


def amount_format(decimals: int, dash_for_zero: bool) -> str:
    base = "0." + "0" * decimals if decimals > 0 else "0"
    return f'{base};-{base};"-"' if dash_for_zero else f"{base};-{base}"


def output_path(cfg: Config, run_date: date) -> Path:
    return cfg.output_dir / f"Fund Available Summary Report {fmt_output_date(run_date)}.xlsx"


def write_report(rows: Sequence[ReportRow], target: Path) -> Path:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = OUTPUT_SHEET_NAME
    for col, header in enumerate(OUTPUT_HEADERS, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.font, cell.fill = HEADER_FONT, HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for offset, row in enumerate(rows):
        r = offset + 2
        ws.cell(r, 1, row.code).font = FONT
        ws.cell(r, 2, row.scheme).font = FONT
        amounts = ((3, row.gross, False), (4, row.redemption, True), (5, row.additional, True),
                   (6, row.switch_in, True), (7, row.switch_out, True))
        for col, amount, dash in amounts:
            cell = ws.cell(r, col, float(amount.value))
            cell.number_format = amount_format(amount.decimals, dash)
            cell.font = FONT
        net = ws.cell(r, 8, f"=ROUND((C{r}-D{r}-E{r}+F{r}-G{r})/{int(CRORE_DIVISOR)},2)")
        net.number_format, net.font = "0.00", FONT

    for col, width in zip("ABCDEFGH", (16, 48, 20, 20, 20, 20, 20, 22)):
        ws.column_dimensions[col].width = width
    ws.row_dimensions[1].height = 30
    ws.freeze_panes = "A2"

    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_name(target.name + ".tmp")
    wb.save(temp)
    os.replace(temp, target)                       # atomic on the same volume
    return target


def verify_output(path: Path, expected_rows: int) -> None:
    wb = openpyxl.load_workbook(path)
    ws = wb[OUTPUT_SHEET_NAME]                     # KeyError -> wrong sheet name
    headers = [c.value for c in ws[1]]
    if headers != OUTPUT_HEADERS:
        raise ReportError(f"Output header mismatch: {headers}")
    if ws.max_row - 1 != expected_rows:
        raise ReportError(f"Output row count {ws.max_row - 1} != expected {expected_rows}")
    if expected_rows and not str(ws.cell(2, 8).value).startswith("=ROUND("):
        raise ReportError("NET formula missing in column H.")
    wb.close()

## 13. Orchestrator

`run()` is deliberately thin — it only sequences the functions above in three phases:

1. **Acquire** — find + download + extract every input. Mandatory inputs are checked first, so nothing is written if a required file is missing.
2. **Parse & transform** — read the Excel files and build the report rows.
3. **Write & verify** — produce the output workbook and validate it.

In [ ]:
def run(cfg: Optional[Config] = None) -> Optional[Path]:
    cfg = cfg or CONFIG
    run_date = resolve_run_date(cfg)
    LOG.info("=== Fund Available Summary Report | run date %s (%s) ===", run_date.isoformat(), run_date.strftime("%A"))

    if not is_business_day(run_date):
        LOG.warning("Run date is a weekend - process stopped, no report generated.")
        return None

    source = build_source(cfg)
    index = source.list_files()
    LOG.info("%d .eml files visible in the source.", len(index))
    input_dir = cfg.input_dir / run_date.isoformat() if cfg.dated_input_subfolder else cfg.input_dir

    # ---- Phase 1: acquire ------------------------------------------------------------
    funds_file = acquire_excel(source, index, pat_funds_available(run_date), input_dir, "Funds Available", True)
    mf_red_file = acquire_excel(source, index, pat_redemption(run_date, False), input_dir, "MF Redemption", True)
    sif_red_file = acquire_excel(source, index, pat_redemption(run_date, True), input_dir, "SIF Redemption", False)
    mf_add_file = acquire_excel(source, index, pat_additional(run_date, False), input_dir, "MF Additional", True)
    sif_add_file = acquire_excel(source, index, pat_additional(run_date, True), input_dir, "SIF Additional", False)

    is_provision = lambda name: PROVISION_ATTACHMENT_KEY in name.upper()
    mf_switch_files: List[Path] = []
    sif_switch_files: List[Path] = []
    for slot_date, am_pm in switch_slots(run_date):
        tag = f"{slot_date.isoformat()} {am_pm}"
        mf_path = acquire_excel(source, index, pat_provision(slot_date, am_pm, False), input_dir,
                                f"MF Daily Provision {tag}", True, is_provision)
        mf_switch_files.append(mf_path)
        sif_path = acquire_excel(source, index, pat_provision(slot_date, am_pm, True), input_dir,
                                 f"SIF Daily Provision {tag}", False, is_provision)
        if sif_path:
            sif_switch_files.append(sif_path)

    # ---- Phase 2: parse & transform --------------------------------------------------
    fund_rows = parse_funds_available(funds_file)
    redemption_mf = parse_funding_statement(mf_red_file, "MF Redemption")
    redemption_sif = parse_funding_statement(sif_red_file, "SIF Redemption") if sif_red_file else {}
    additional_mf = parse_funding_statement(mf_add_file, "MF Additional")
    additional_sif = parse_funding_statement(sif_add_file, "SIF Additional") if sif_add_file else {}
    switches_mf = merge_switches([parse_daily_provision(p, "MF Daily Provision") for p in mf_switch_files])
    switches_sif = merge_switches([parse_daily_provision(p, "SIF Daily Provision") for p in sif_switch_files])

    rows = build_report_rows(fund_rows, cfg.sif_scheme_codes, redemption_mf, redemption_sif,
                             additional_mf, additional_sif, switches_mf, switches_sif)

    # ---- Phase 3: write & verify -----------------------------------------------------
    target = write_report(rows, output_path(cfg, run_date))
    verify_output(target, expected_rows=len(rows))
    total_net = sum((compute_net_crores(r) for r in rows), Decimal(0))
    LOG.info("Report written: %s | rows=%d (SIF=%d) | sum of NET (Crs)=%s",
             target, len(rows), sum(r.is_sif for r in rows), total_net)
    return target

## 14. Built-in self-tests

Fast, offline checks of the rules that are easiest to get wrong (file-name matching, `Uploaded` exclusion, Monday/Friday logic, bracket handling, crore rounding, zero display). Run them after any change to the code above.

In [ ]:
def run_self_tests() -> None:
    now = datetime.now(timezone.utc)
    mk = lambda n: SourceFile(n, n, now)
    d = date(2026, 9, 25)                                            # Friday

    files = [mk("Redemption Payout - Funding Statement for the Payment dt. 25.09.2026.eml"),
             mk("Uploaded Redemption Payout - Funding Statement for the Payment dt. 25.09.2026.eml"),
             mk("ISIF_Redemption Payout - Funding Statement for the Payment dt. 25.09.2026.eml"),
             mk("RE: Funds Available as of 25 Sep 2026.eml"),
             mk("Uploaded Funds Available as of 25 Sep 2026.eml")]
    assert find_eml(files, pat_redemption(d, False)).name.startswith("Redemption Payout")
    assert find_eml(files, pat_redemption(d, True)).name.startswith("ISIF_")
    assert find_eml(files[1:2], pat_redemption(d, False)) is None                # 'Uploaded' never used
    assert find_eml(files, pat_funds_available(d)).name.startswith("RE:")
    assert find_eml(files, pat_redemption(date(2026, 9, 26), False)) is None      # wrong date

    prov = [mk("ISIF Daily Provision and Net Off Intra Switches file for the trade dated 27-SEP-2026.AM26092027033255.eml"),
            mk("Daily Provision and Net Off Intra Switches file for the trade dated 27-SEP-2026.AM.eml")]
    assert find_eml(prov, pat_provision(date(2026, 9, 27), "AM", True)).name.startswith("ISIF")
    assert find_eml(prov, pat_provision(date(2026, 9, 27), "AM", False)).name.startswith("Daily")

    assert switch_slots(date(2026, 9, 28)) == [(date(2026, 9, 25), "PM"), (date(2026, 9, 27), "AM")]   # Monday
    assert switch_slots(date(2026, 9, 29)) == [(date(2026, 9, 28), "AM")]                               # Tuesday
    assert is_business_day(date(2026, 9, 25)) and not is_business_day(date(2026, 9, 26))

    assert to_decimal("(8134535.56)") == Decimal("-8134535.56")
    assert to_decimal(8134535.56) == Decimal("8134535.56")
    assert to_decimal("1,234.50") == Decimal("1234.50") and to_decimal(None) == 0 and to_decimal("-") == 0
    assert amount_format(2, True) == '0.00;-0.00;"-"'

    row = ReportRow("X", "X", Amount(Decimal("46500546.43")), ZERO, ZERO, ZERO, ZERO, False)
    assert compute_net_crores(row) == Decimal("4.65")
    print("All self-tests passed.")


run_self_tests()

## 15. Run

Executes the whole process. Non-business days stop gracefully; missing mandatory inputs or malformed files raise an exception (the notebook run fails, so the scheduler can alert).

In [ ]:
OUTPUT_FILE = run()
OUTPUT_FILE

## 16. Pseudo code

```text
PROCESS FundAvailableSummaryReport

CONFIG:
    Input/ and Output/ folders next to the notebook (create if missing)
    SIF_SCHEME_CODES = [ ... ]                       # array of SIF MFUND CODES
    EML source = OneDrive folder (Microsoft Graph)   # local folder for dry-run

T = today (Asia/Kolkata)
IF T is Saturday or Sunday THEN log and STOP (no output)

emlList = list all *.eml names in the OneDrive folder
NEVER consider any eml whose name starts with "Uploaded"

FUNCTION findEml(namePrefix):
    match names that BEGIN with namePrefix (optionally after RE:/FW:)   # keeps ISIF_ apart from MF
    if several -> take latest modified
FUNCTION getExcel(eml, filter):
    attachments = all attachments of eml
    IF a .zip exists THEN unzip -> take the Excel file inside      # exactly one expected
    ELSE take the Excel attachment directly
    apply filter (e.g. name contains DAILY_PROVISION)
    IF count != 1 THEN error
    save into Input/<T>/ and return path

# ---------- 1. acquire ----------
fundsFile   = getExcel(findEml("Funds Available as of dd MMM yyyy"))                      # missing -> ERROR + STOP
mfRedFile   = getExcel(findEml("Redemption Payout - Funding Statement for the Payment dt. dd.MM.yyyy"))   # missing -> ERROR + STOP
sifRedFile  = getExcel(findEml("ISIF_Redemption Payout - Funding Statement ... dd.MM.yyyy"))              # missing -> continue
mfAddFile   = getExcel(findEml("Redemption Payout - Separate Statement ... dd.MM.yyyy"))                   # missing -> ERROR + STOP
sifAddFile  = getExcel(findEml("ISIF_Redemption Payout - Separate Statement ... dd.MM.yyyy"))              # missing -> continue

IF T is Monday THEN slots = [ (Friday, PM), (Sunday, AM) ]
ELSE                     slots = [ (T-1, AM) ]
FOR each slot:
    mfSwitchFiles  += getExcel(findEml("Daily Provision and Net Off Intra Switches file for the trade dated dd-MMM-yyyy.<AM|PM>"),
                               attachment name contains DAILY_PROVISION)                # missing -> ERROR + STOP
    sifSwitchFiles += same with "ISIF Daily Provision ..." prefix                        # missing -> skip

# ---------- 2. parse ----------
fundRows = read Funds Available sheet: A=MFUND CODE, B=SCHEME, C=GROSS
           IF C is like (123.45) THEN value = -123.45 (same decimals)

FUNCTION readFunding(file):                          # FUNDING sheet
    FOR each row where column A starts with "MFUND CODE":        # table start
        totalRow = next row where column A = "Total Amount"
        FOR each column B..N: dict[schemeCode in header row] = value in totalRow   # as-is, no format change
    RETURN dict

FUNCTION readProvision(file):
    header row = row where column C = MFUND_CODE
    FOR each row: dict[C] = (F = SWITCH_IN, G = SWITCH_OUT)      # duplicate codes summed
FUNCTION merge(files): sum SWITCH_IN and SWITCH_OUT per code across files   # Monday: Fri PM + Sun AM

redMF, redSIF = readFunding(mfRedFile), readFunding(sifRedFile) or {}
addMF, addSIF = readFunding(mfAddFile), readFunding(sifAddFile) or {}
swMF,  swSIF  = merge(readProvision each mfSwitchFile), merge(readProvision each sifSwitchFile)

# ---------- 3. build ----------
SORT fundRows ascending by MFUND CODE
FOR each fundRow:
    isSIF = code IN SIF_SCHEME_CODES
    REDEMPTION = (isSIF ? redSIF : redMF)[code]  or 0   # 0 is displayed as "-"
    ADDITIONAL = (isSIF ? addSIF : addMF)[code]  or 0
    (SWITCH IN, SWITCH OUT) = (isSIF ? swSIF : swMF)[code] or (0, 0)
    NET = ROUND( (GROSS - REDEMPTION - ADDITIONAL + SWITCH IN - SWITCH OUT) / 10000000, 2 )   # Excel formula, 4,65,00,546.43 -> 4.65

# ---------- 4. write ----------
create Output/Fund Available Summary Report dd-MMM-yyyy.xlsx
    sheet "Fund Available Summary Report"
    columns: MFUND CODE | SCHEME | GROSS | REDEMPTION | ADDITIONAL | SWITCH IN | SWITCH OUT | NET (Amount in Crs)
    zero values in D:G display as "-"
verify file (sheet, headers, row count, formula) and log summary
END
```